# VDP vs digest2 — ROC Comparison

Head-to-head comparison of the VDP probability-map classifier vs digest2 on the same S3M objects in the ecliptic strip (|β| < 3°, epoch 2025-03-21), restricted to the VDP map's operating region (opposition ±30°, mag 14–26).

**Fast path (recommended):** Run section 2 onward — loads the pre-computed parquet.

**Full re-run:** Cell below with `%run` — takes ~30 min (propagation + digest2).

In [ ]:
# Optional: re-run the full pipeline (slow — ~30 min)
# %run run_digest2_comparison.py

## 1. Setup

In [1]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))
import velocity_density_pipeline as vdp

_HERE = os.path.dirname(os.path.abspath('__file__'))
PARQUET = os.path.join(_HERE, 's3m_digest2_comparison.parquet')
PROB_MAPS = os.path.join(_HERE, 'prob_maps_2025-03-21.npz')
FIG_OUT  = os.path.join(_HERE, 'roc_comparison_vdp_digest2.png')

%matplotlib inline
plt.rcParams.update({'figure.dpi': 130, 'font.size': 11})

## 2. Load results

In [2]:
df = pd.read_parquet(PARQUET)
print(f'{len(df):,} objects')
print(df['true_population'].value_counts().to_string())

pops = sorted(df['true_population'].unique())
colors = {'NEO': 'tab:blue', 'MBA': 'tab:orange', 'TNO': 'tab:green', 'Trojans': 'tab:red'}
df.head()

2,658 objects
true_population
MBA    2296
NEO     362


,true_population,lam_deg,vlam,vbeta,mag_app,H,P_NEO_vdp,P_NEO_d2
0,NEO,188.001674,-0.295547,0.008031,20.306694,16.442,0.420232,0.10
1,NEO,204.072011,-0.207662,0.036626,21.942001,16.528,0.000058,0.00
2,NEO,208.902898,-0.247018,-0.088659,21.880168,17.508,0.000176,0.73
3,NEO,192.049639,-0.232906,0.037024,22.740620,17.659,0.000277,0.00
4,NEO,205.818622,-0.225690,-0.038671,22.347825,17.747,0.000734,0.01


In [ ]:
# Re-score VDP from the probability maps (in case you want fresh scores)
pm = vdp.ProbMapSet.from_npz(PROB_MAPS)
probs = pm.score_visible(df.vlam.to_numpy(), df.vbeta.to_numpy(), df.mag_app.to_numpy())
df['P_NEO_vdp'] = probs['NEO']
df['P_MBA_vdp'] = probs['MBA']
df['P_TNO_vdp'] = probs['TNO']
df['P_Troj_vdp'] = probs['Trojans']
print('VDP P_NEO range:', round(float(df.P_NEO_vdp.min()), 4), '–', round(float(df.P_NEO_vdp.max()), 4))
print('sum check (should be ≤1):', (df.P_NEO_vdp + df.P_MBA_vdp + df.P_TNO_vdp + df.P_Troj_vdp).describe())

## 3. Score distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
pops = df['true_population'].unique()
colors = {'NEO': 'tab:blue', 'MBA': 'tab:orange', 'TNO': 'tab:green', 'Trojans': 'tab:red'}

for ax, (col, label) in zip(axes, [('P_NEO_vdp', 'VDP P_NEO'), ('P_NEO_d2', 'digest2 score')]):
    for pop in pops:
        sub = df.loc[df.true_population == pop, col]
        ax.hist(sub, bins=50, alpha=0.55, label=pop, color=colors.get(pop),
                density=True, range=(0, 1))
    ax.set_xlabel(label); ax.set_ylabel('Density')
    ax.set_title(f'{label} by population')
    ax.legend(fontsize=9)

plt.tight_layout()
plt.show()
plt.close(fig)

## 4. ROC curves

In [4]:
is_neo = (df['true_population'] == 'NEO').to_numpy()
N_neo  = is_neo.sum()
N_non  = (~is_neo).sum()
print(f'N_NEO={N_neo}  N_non-NEO={N_non}')

thresholds = np.linspace(0, 1, 401)

def compute_roc(scores, is_neo, thresholds):
    comp = np.empty(len(thresholds))
    cont = np.empty(len(thresholds))
    for j, t in enumerate(thresholds):
        above = scores > t
        tp = above[is_neo].sum()
        fp = above[~is_neo].sum()
        comp[j] = tp / max(N_neo, 1)
        cont[j] = fp / max(tp + fp, 1)
    return comp, cont

def best_f1(comp, cont, thr):
    pur = 1 - cont
    f1  = np.where(pur + comp > 0, 2*pur*comp/(pur + comp + 1e-12), 0.0)
    i   = np.argmax(f1)
    return thr[i], comp[i], cont[i], f1[i]

comp_v, cont_v = compute_roc(df['P_NEO_vdp'].to_numpy(), is_neo, thresholds)
comp_d, cont_d = compute_roc(df['P_NEO_d2'].to_numpy(),  is_neo, thresholds)

tv, cv, nv, fv = best_f1(comp_v, cont_v, thresholds)
td, cd, nd, fd = best_f1(comp_d, cont_d, thresholds)

print(f'VDP     best: t={tv:.3f}  completeness={cv:.1%}  contamination={nv:.1%}  F1={fv:.3f}')
print(f'Digest2 best: t={td:.3f}  completeness={cd:.1%}  contamination={nd:.1%}  F1={fd:.3f}')

N_NEO=362  N_non-NEO=2296
VDP     best: t=0.020  completeness=75.4%  contamination=3.5%  F1=0.847
Digest2 best: t=0.090  completeness=75.4%  contamination=6.8%  F1=0.834


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(comp_v*100, cont_v*100, lw=2.2, color='tab:blue',   label='VDP (this work)')
ax.plot(comp_d*100, cont_d*100, lw=2.2, color='tab:orange', linestyle='--', label='digest2')
ax.scatter([cv*100], [nv*100], color='tab:blue',   s=120, zorder=5, label=f'VDP best  F1={fv:.2f}')
ax.scatter([cd*100], [nd*100], color='tab:orange', s=120, zorder=5, marker='s', label=f'd2 best   F1={fd:.2f}')
ax.set_xlabel('Completeness (%)'); ax.set_ylabel('Contamination (%)')
ax.set_title(f'ROC: VDP vs digest2\n(opposition ±30°, mag 14–26, N_NEO={N_neo})', fontsize=10)
ax.legend(fontsize=9); ax.grid(alpha=0.3)
ax.set_xlim(0, 100); ax.set_ylim(0, 100)

ax = axes[1]
ax.plot(thresholds, comp_v*100, lw=2,   color='tab:blue',   label='VDP completeness')
ax.plot(thresholds, cont_v*100, lw=2,   color='tab:blue',   linestyle=':', label='VDP contamination')
ax.plot(thresholds, comp_d*100, lw=2,   color='tab:orange', linestyle='--', label='d2 completeness')
ax.plot(thresholds, cont_d*100, lw=2,   color='tab:orange', linestyle='-.', label='d2 contamination')
ax.axvline(tv, color='tab:blue',   alpha=0.4, lw=1.2)
ax.axvline(td, color='tab:orange', alpha=0.4, lw=1.2)
ax.set_xlabel('Score threshold'); ax.set_ylabel('Completeness / Contamination (%)')
ax.set_title('Threshold vs. Performance', fontsize=10)
ax.legend(fontsize=9); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(FIG_OUT, dpi=150, bbox_inches='tight')
print(f'Saved: {FIG_OUT}')
plt.show()
plt.close(fig)

## 5. Per-population score breakdown

In [6]:
for pop in sorted(df.true_population.unique()):
    sub = df[df.true_population == pop]
    vdp_med = sub.P_NEO_vdp.median()
    d2_med  = sub.P_NEO_d2.median()
    vdp_hi  = (sub.P_NEO_vdp > tv).mean()
    d2_hi   = (sub.P_NEO_d2  > td).mean()
    print(f'{pop:10s}  n={len(sub):4d}  '
          f'VDP median={vdp_med:.3f}  above_thr={vdp_hi:.1%}  '
          f'| d2 median={d2_med:.3f}  above_thr={d2_hi:.1%}')

MBA         n=2296  VDP median=0.000  above_thr=0.4%  | d2 median=0.000  above_thr=0.9%
NEO         n= 362  VDP median=0.832  above_thr=75.4%  | d2 median=1.000  above_thr=75.4%


## 6. Explore: score vs. observable properties

In [ ]:
pops = sorted(df['true_population'].unique())
colors = {'NEO': 'tab:blue', 'MBA': 'tab:orange', 'TNO': 'tab:green', 'Trojans': 'tab:red'}

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, (xcol, xlabel) in zip(axes, [('vlam', 'v_λ (deg/day)'), ('mag_app', 'mag_app')]):
    for pop in pops:
        sub = df[df.true_population == pop]
        ax.scatter(sub[xcol], sub['P_NEO_vdp'], s=4, alpha=0.3,
                   label=pop, color=colors.get(pop))
    ax.axhline(tv, color='k', lw=0.8, linestyle='--', label=f'VDP threshold ({tv:.3f})')
    ax.set_xlabel(xlabel); ax.set_ylabel('VDP P_NEO')
    ax.legend(fontsize=8, markerscale=3)

plt.tight_layout()
plt.show()
plt.close(fig)